# **PIP-Water**



##  1) Imports e dependências

In [1]:
%pip install -q pandas requests python-dotenv pymongo pymysql certifi clts_pcp
import hashlib
import json
import os
import re
import random
import smtplib
import socket
import ssl
import time
from datetime import datetime, timedelta, timezone
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from pathlib import Path

import certifi
import clts_pcp as clts
import pandas as pd
import requests

try:
    from pymongo import MongoClient, UpdateOne
except Exception:
    MongoClient = None
    UpdateOne = None

try:
    import pymysql
except Exception:
    pymysql = None

try:
    import psycopg2
except Exception:
    psycopg2 = None

def env_bool(name: str, default: str = 'false') -> bool:
    return os.getenv(name, default).strip().lower() in {'1', 'true', 'yes', 'on'}


def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return google.colab is not None
    except Exception:
        return False


def is_render() -> bool:
    return bool(os.getenv('RENDER') or os.getenv('RENDER_SERVICE_ID') or os.getenv('RENDER_EXTERNAL_URL'))


def detect_runtime() -> str:
    if is_colab():
        return 'google-colab'
    if is_render():
        return 'render'
    if os.getenv('AIRFLOW_HOME') or os.getenv('AIRFLOW_CTX_DAG_ID'):
        return 'airflow'
    if os.getenv('FLASK_ENV') or os.getenv('WERKZEUG_RUN_MAIN'):
        return 'flask'
    return 'local'


def env_or_colab_secret(name: str, default: str = '') -> str:
    value = (os.getenv(name) or '').strip()
    if value:
        return value

    if not is_colab():
        return default

    try:
        from google.colab import userdata  # type: ignore
        raw = userdata.get(name)
        if not raw:
            return default
        raw_text = str(raw).strip()
        try:
            payload = json.loads(raw_text)
            if isinstance(payload, dict):
                for key in ('value', 'uri', 'token', 'key', 'password'):
                    candidate = payload.get(key)
                    if candidate:
                        return str(candidate).strip()
        except Exception:
            pass
        return raw_text
    except Exception:
        return default


if not is_colab():
    try:
        from dotenv import find_dotenv, load_dotenv
        dotenv_file = find_dotenv('.env', usecwd=True)
        if dotenv_file:
            load_dotenv(dotenv_file, override=True)
        else:
            # Fallback for cases where notebook cwd is inside a subfolder.
            for candidate in (Path.cwd() / '.env', Path.cwd().parent / '.env'):
                if candidate.exists():
                    load_dotenv(candidate, override=True)
                    break
    except Exception:
        pass


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


##  2) Configuração

In [2]:
mongo_uri_value = env_or_colab_secret('MONGO_URI', '').strip()
tidb_host_value = env_or_colab_secret('TIDB_HOST', '').strip()
tidb_port_value = int(env_or_colab_secret('TIDB_PORT', '4000'))
tidb_user_value = env_or_colab_secret('TIDB_USER', '').strip()
tidb_password_value = env_or_colab_secret('TIDB_PASSWORD', '').strip()
tidb_database_value = env_or_colab_secret('TIDB_DATABASE', 'pi_water').strip()
tidb_table_value = env_or_colab_secret('TIDB_TABLE', 'water_records').strip()
tidb_ca_path_value = env_or_colab_secret('TIDB_CA_PATH', '').strip()
cratedb_host_value = env_or_colab_secret('CRATEDB_HOST', '').strip()
cratedb_port_value = int(env_or_colab_secret('CRATEDB_PORT', '5432'))
cratedb_user_value = env_or_colab_secret('CRATEDB_USER', '').strip()
cratedb_password_value = env_or_colab_secret('CRATEDB_PASSWORD', '').strip()
cratedb_database_value = env_or_colab_secret('CRATEDB_DATABASE', 'crate').strip()
cratedb_table_value = env_or_colab_secret('CRATEDB_TABLE', 'water_records').strip()
cratedb_sslmode_value = env_or_colab_secret('CRATEDB_SSLMODE', 'require').strip()


def _local_default(value_if_local: str, value_if_colab: str) -> str:
    return value_if_local if not is_colab() else value_if_colab


CONFIG = {
    'repo_owner': os.getenv('REPO_OWNER', 'pedroccpimenta'),
    'repo_name': os.getenv('REPO_NAME', 'datafiles'),
    'repo_folder': os.getenv('REPO_FOLDER', 'aqualog'),
    'repo_branch': os.getenv('REPO_BRANCH', 'master'),
    'request_timeout': int(os.getenv('REQUEST_TIMEOUT', _local_default('15', '30'))),
    'days_back': int(os.getenv('DAYS_BACK', '0')),
    'start_date': os.getenv('START_DATE', datetime.now(timezone.utc).strftime('%Y-%m-%d')),
    'output_folder': os.getenv('OUTPUT_FOLDER', '/content/output_water' if is_colab() else './output_water'),
    'json_file_urls': os.getenv('JSON_FILE_URLS', ''),
    'local_json_dir': os.getenv('LOCAL_JSON_DIR', ''),
    'repo_json_files': os.getenv('REPO_JSON_FILES', ''),
    'verbose': env_bool('VERBOSE', 'true'),
    'email_enabled': env_bool('EMAIL_ENABLED', 'false'),
    'email_smtp_host': os.getenv('EMAIL_SMTP_HOST', 'smtp.gmail.com'),
    'email_smtp_port': int(os.getenv('EMAIL_SMTP_PORT', '587')),
    'email_from': os.getenv('EMAIL_FROM', ''),
    'email_to': os.getenv('EMAIL_TO', ''),
    'email_username': os.getenv('EMAIL_USERNAME', ''),
    'email_password': os.getenv('EMAIL_PASSWORD', ''),
    'email_backend': os.getenv('EMAIL_BACKEND', 'brevo' if is_render() else 'smtp').strip(),
    'brevo_api_key': os.getenv('BREVO_API_KEY', '').strip(),
    'brevo_sender_name': os.getenv('BREVO_SENDER_NAME', '').strip(),
    'brevo_sender_email': os.getenv('BREVO_SENDER_EMAIL', '').strip(),
    'github_secret_name': os.getenv('GITHUB_SECRET_NAME', '').strip(),

    'mongo_uri': mongo_uri_value,
    'mongo_enabled': env_bool('MONGO_ENABLED', 'false') or bool(mongo_uri_value),
    'mongo_db': os.getenv('MONGO_DB', 'pi_water').strip(),
    'mongo_collection': os.getenv('MONGO_COLLECTION', 'water_records').strip(),
    'mongo_app_name': os.getenv('MONGO_APP_NAME', 'pip-water').strip(),
    'mongo_server_selection_timeout_ms': int(os.getenv('MONGO_SERVER_SELECTION_TIMEOUT_MS', _local_default('4000', '10000'))),

    'tidb_host': tidb_host_value,
    'tidb_port': tidb_port_value,
    'tidb_user': tidb_user_value,
    'tidb_password': tidb_password_value,
    'tidb_database': tidb_database_value,
    'tidb_table': tidb_table_value,
    'tidb_ca_path': tidb_ca_path_value,
    'tidb_connect_timeout': int(env_or_colab_secret('TIDB_CONNECT_TIMEOUT', _local_default('4', '10'))),
    'tidb_enabled': env_bool('TIDB_ENABLED', 'false') or bool(
        tidb_host_value and tidb_user_value and tidb_password_value and tidb_database_value and tidb_table_value
    ),

    'cratedb_host': cratedb_host_value,
    'cratedb_port': cratedb_port_value,
    'cratedb_user': cratedb_user_value,
    'cratedb_password': cratedb_password_value,
    'cratedb_database': cratedb_database_value,
    'cratedb_table': cratedb_table_value,
    'cratedb_sslmode': cratedb_sslmode_value,
    'cratedb_enabled': env_bool('CRATEDB_ENABLED', 'false') or bool(
        cratedb_host_value and cratedb_user_value and cratedb_password_value and cratedb_database_value and cratedb_table_value
    ),

    'anomaly_threshold_ratio': float(os.getenv('ANOMALY_THRESHOLD_RATIO', '0.8')),
    'anomaly_lookback_days': int(os.getenv('ANOMALY_LOOKBACK_DAYS', '2')),
    'max_files': int(os.getenv('MAX_FILES', '0')),
    'repo_sample_size': int(os.getenv('REPO_SAMPLE_SIZE', '30')),
    'skip_db_writes': env_bool('SKIP_DB_WRITES', 'false'),
}

# clts-style profiling context
_tstart = clts.getts()
clts.elapt.clear()
_hostname = socket.gethostname()
try:
    _ip = requests.get('https://api.ipify.org', timeout=5).text
except Exception:
    _ip = 'unknown'
_user = os.getenv('D5_USER') or os.getenv('USERNAME') or os.getenv('USER') or 'PI'
_script = 'pi_water.ipynb'
_channel = 'WATER'
_destination = f"{CONFIG['repo_owner']}/{CONFIG['repo_name']}/{CONFIG['repo_folder']}"
context = f'{_hostname} ({_ip}) | {_user} | {_channel} | {_script} | {_destination}'
clts.setcontext(context)
clts.elapt[f'running notebook: {_script}'] = clts.deltat(_tstart)

send_mail = CONFIG['email_enabled']
email_addresses = [item.strip() for item in CONFIG['email_to'].split(',') if item.strip()]

print('Runtime:', detect_runtime())
print('Context:', context)
print('Repo target:', f"{CONFIG['repo_owner']}/{CONFIG['repo_name']}/{CONFIG['repo_folder']}")
print('Output folder:', CONFIG['output_folder'])
print('Mongo enabled:', CONFIG['mongo_enabled'])
print('Mongo URI set:', bool(CONFIG['mongo_uri']))
print('TiDB enabled:', CONFIG['tidb_enabled'])
print('TiDB host set:', bool(CONFIG['tidb_host']))
print('TiDB database:', CONFIG['tidb_database'])
print('CrateDB enabled:', CONFIG['cratedb_enabled'])
print('CrateDB host set:', bool(CONFIG['cratedb_host']))
print('CrateDB database:', CONFIG['cratedb_database'])
print('REQUEST_TIMEOUT:', CONFIG['request_timeout'])
print('MAX_FILES:', CONFIG['max_files'])
print('REPO_SAMPLE_SIZE:', CONFIG['repo_sample_size'])
print('SKIP_DB_WRITES:', CONFIG['skip_db_writes'])
print('send_mail:', send_mail)
print('email_addresses:', email_addresses)

Runtime: local
Context: perry (149.90.118.154) | henrique | WATER | pi_water.ipynb | pedroccpimenta/datafiles/aqualog
Repo target: pedroccpimenta/datafiles/aqualog
Output folder: ./output_water
Mongo enabled: True
Mongo URI set: True
TiDB enabled: True
TiDB host set: True
TiDB database: pi_water
CrateDB enabled: True
CrateDB host set: True
CrateDB database: crate
REQUEST_TIMEOUT: 30
MAX_FILES: 0
REPO_SAMPLE_SIZE: 30
SKIP_DB_WRITES: False
send_mail: True
email_addresses: ['henriqueperryg@gmail.com']


##  3) Token GitHub

In [3]:
def load_colab_secret(secret_name: str):
    if not is_colab():
        return None
    try:
        from google.colab import userdata  # type: ignore
        raw = userdata.get(secret_name)
        if not raw:
            return None
        try:
            return json.loads(raw)
        except Exception:
            return {'token': str(raw).strip()}
    except Exception:
        return None


def get_github_token() -> str | None:
    token = (os.getenv('GITHUB_TOKEN') or '').strip()
    if token:
        return token

    candidates = []
    if CONFIG['github_secret_name']:
        candidates.append(CONFIG['github_secret_name'])
    candidates.extend([
        'GITHUB_TOKEN',
        'github_token',
        'TO-github.json',
        'TO-github_token.json',
        'github_token.json',
    ])

    for secret_name in candidates: #tenta um por um
        payload = load_colab_secret(secret_name)
        if not payload:
            continue
        value = payload.get('key') or payload.get('token')
        if value:
            return str(value).strip()

    return None


def headers(token: str | None) -> dict[str, str]:
    h = {'Accept': 'application/vnd.github+json'}
    if token:
        h['Authorization'] = f'Bearer {token}'
    return h


TOKEN = get_github_token()
print('GitHub token presente:', bool(TOKEN))

GitHub token presente: True


## 4) Descoberta de ficheiros 

In [4]:
def raw_url(file_path: str, branch: str) -> str:
    return f"https://raw.githubusercontent.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}/{branch}/{file_path}"


def branch_candidates() -> list[str]:
    preferred = CONFIG['repo_branch'].strip()
    values = [preferred, 'main', 'master']
    unique: list[str] = []
    for value in values:
        if value and value not in unique:
            unique.append(value)
    return unique


def list_from_json_file_urls() -> list[dict[str, str]]:
    items = [u.strip() for u in CONFIG['json_file_urls'].split(',') if u.strip()]
    return [{'name': Path(item).name, 'source': item} for item in items]


def list_from_local_dir() -> list[dict[str, str]]:
    local_dir = CONFIG['local_json_dir'].strip()
    if not local_dir:
        return []
    folder = Path(local_dir)
    if not folder.exists():
        return []
    return [{'name': p.name, 'source': str(p)} for p in sorted(folder.glob('*.json'))]


def list_from_known_names() -> list[dict[str, str]]:
    names = [n.strip() for n in CONFIG['repo_json_files'].split(',') if n.strip()]
    chosen_branch = branch_candidates()[0]
    return [
        {'name': Path(name).name, 'source': raw_url(f"{CONFIG['repo_folder']}/{name}", chosen_branch)}
        for name in names
    ]


def list_from_github(token: str | None) -> list[dict[str, str]]:
    last_error: str | None = None

    for branch in branch_candidates():
        api_url = f"https://api.github.com/repos/{CONFIG['repo_owner']}/{CONFIG['repo_name']}/contents/{CONFIG['repo_folder']}"
        response = requests.get(
            api_url,
            headers=headers(token),
            params={'ref': branch},
            timeout=CONFIG['request_timeout'],
        )

        if response.ok:
            payload = response.json()
            files = [
                {'name': item['name'], 'source': item['download_url']}
                for item in payload
                if item.get('type') == 'file' and str(item.get('name', '')).endswith('.json')
            ]
            if files:
                return files

        last_error = f'API {branch}: {response.status_code} {response.reason}'

        tree_url = f"https://github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}/tree/{branch}/{CONFIG['repo_folder']}"
        page = requests.get(tree_url, timeout=CONFIG['request_timeout'])
        if page.ok:
            pattern = (
                rf'href="/{re.escape(CONFIG["repo_owner"])}'
                rf'/{re.escape(CONFIG["repo_name"])}'
                rf'/blob/{re.escape(branch)}/{re.escape(CONFIG["repo_folder"])}'
                rf'/([^\"]+?\.json)"'
            )
            matches = sorted(set(re.findall(pattern, page.text)))
            if matches:
                return [
                    {'name': Path(name).name, 'source': raw_url(f"{CONFIG['repo_folder']}/{name}", branch)}
                    for name in matches
                ]

    if last_error:
        raise RuntimeError(
            f'GitHub listing failed for branches {branch_candidates()}: {last_error}. '
            'Confirma token, branch e permissao read no repositorio privado.'
        )
    return []


def extract_date_from_name(file_name: str):
    match = re.search(r'(20\d{6})', file_name)
    if not match:
        return None
    try:
        return datetime.strptime(match.group(1), '%Y%m%d').date()
    except ValueError:
        return None


def apply_date_window(files: list[dict[str, str]]) -> list[dict[str, str]]:
    days_back = CONFIG['days_back']
    if days_back <= 0:
        return files

    end_date = datetime.strptime(CONFIG['start_date'], '%Y-%m-%d').date()
    start_date = end_date - timedelta(days=days_back)

    selected = []
    for file_info in files:
        file_date = extract_date_from_name(file_info['name'])
        if file_date is None or (start_date <= file_date <= end_date):
            selected.append(file_info)
    return selected


def limit_repo_files_randomly(files: list[dict[str, str]]) -> list[dict[str, str]]:
    sample_size = int(CONFIG.get('repo_sample_size', 30) or 30)
    if sample_size <= 0 or len(files) <= sample_size:
        return files
    return random.sample(files, sample_size)

## 4.1) Descoberta de ficheiros JSON

## 5) Transformação, persistência e outputs

Esta célula lê os JSONs, normaliza os registos, limpa os dados e prepara os outputs e as funções de persistência para as bases de dados.

In [5]:
# Transformacao, persistencia e outputs

def read_json_source(source: str, token: str | None):
    if source.startswith('http://') or source.startswith('https://'):
        response = requests.get(source, headers=headers(token), timeout=CONFIG['request_timeout'])
        response.raise_for_status()
        return response.json()

    path = Path(source)
    if not path.exists():
        raise FileNotFoundError(f'JSON file not found: {source}')

    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)


def normalize_records(data, source_name: str, run_id: str) -> list[dict]:
    ingested_at = datetime.now(timezone.utc).isoformat()
    if isinstance(data, list):
        rows = data
    elif isinstance(data, dict):
        rows = [data]
    else:
        rows = [{'value': data}]

    out = []
    for row in rows:
        record = dict(row) if isinstance(row, dict) else {'value': row}
        record['_source_file'] = source_name
        record['_ingested_at'] = ingested_at
        record['_run_id'] = run_id
        out.append(record)
    return out


def light_clean(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    dedup_key = df.apply(lambda row: json.dumps(row.to_dict(), sort_keys=True, default=str), axis=1)
    df = df.loc[~dedup_key.duplicated()].copy()

    for col in ['timestamp', 'time', 'datetime', 'datahora', 'date']:
        if col in df.columns:
            parsed = pd.to_datetime(df[col], errors='coerce', utc=True)
            df[col] = parsed.dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    return df


def save_outputs(df: pd.DataFrame):
    output_dir = Path(CONFIG['output_folder'])
    output_dir.mkdir(parents=True, exist_ok=True)

    json_path = output_dir / 'water_data_raw.json'
    csv_path = output_dir / 'water_data_clean.csv'
    result_path = output_dir / 'water_result.json'

    df.to_json(json_path, orient='records', force_ascii=False)
    df.to_csv(csv_path, index=False)

    return {
        'json_path': str(json_path),
        'csv_path': str(csv_path),
        'result_path': str(result_path),
    }


def _doc_hash(document: dict) -> str:
    payload = json.dumps(document, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def _stable_record_identity(record: dict) -> str:
    identity = dict(record)
    for volatile_key in ('_ingested_at', '_run_id', '_saved_at', '_last_run_id'):
        identity.pop(volatile_key, None)
    return _doc_hash(identity)


def _utc_sql_datetime(value):
    if isinstance(value, datetime):
        return value.replace(tzinfo=None)
    if not value:
        return datetime.utcnow()
    try:
        parsed = datetime.fromisoformat(str(value).replace('Z', '+00:00'))
        if parsed.tzinfo is not None:
            return parsed.astimezone(timezone.utc).replace(tzinfo=None)
        return parsed
    except Exception:
        return datetime.utcnow()


def _sanitize_mongo_uri_for_colab(uri: str) -> str:
    if not uri or not is_colab() or '?' not in uri:
        return uri

    base, query = uri.split('?', 1)
    keep = []
    remove_keys = {'tlscafile', 'ssl_ca_certs', 'tlscertificatekeyfile'}
    for item in query.split('&'):
        key = item.split('=', 1)[0].strip().lower()
        if key in remove_keys:
            continue
        keep.append(item)

    return f"{base}?{'&'.join(keep)}" if keep else base


def save_to_mongodb(df: pd.DataFrame, run_id: str) -> dict:
    if not CONFIG['mongo_enabled']:
        return {'enabled': False, 'status': 'disabled'}

    if not CONFIG['mongo_uri']:
        return {'enabled': True, 'status': 'error', 'error': 'MONGO_URI is empty.'}

    if MongoClient is None or UpdateOne is None:
        return {
            'enabled': True,
            'status': 'error',
            'error': 'pymongo is not installed. Install dependencies from requirements.txt.',
        }

    records = json.loads(df.to_json(orient='records', force_ascii=False, date_format='iso'))
    if not records:
        return {'enabled': True, 'status': 'ok', 'inserted': 0, 'duplicates': 0, 'total': 0}

    operations = []
    for record in records:
        doc = dict(record)
        doc['_saved_at'] = datetime.now(timezone.utc).isoformat()
        doc['_last_run_id'] = run_id
        doc_id = _stable_record_identity(doc)
        doc['_id'] = doc_id
        operations.append(UpdateOne({'_id': doc_id}, {'$setOnInsert': doc}, upsert=True))

    client = None
    try:
        mongo_uri = _sanitize_mongo_uri_for_colab(CONFIG['mongo_uri'])
        client_kwargs = {
            'appname': CONFIG['mongo_app_name'] or 'pip-water',
            'serverSelectionTimeoutMS': CONFIG['mongo_server_selection_timeout_ms'],
        }

        try:
            if certifi is not None:
                ca_path = certifi.where()
                if ca_path and os.path.exists(ca_path):
                    client_kwargs['tlsCAFile'] = ca_path
        except Exception:
            pass

        client = MongoClient(mongo_uri, **client_kwargs)
        client.admin.command('ping')
        collection = client[CONFIG['mongo_db']][CONFIG['mongo_collection']]
        write_result = collection.bulk_write(operations, ordered=False)

        inserted = int(getattr(write_result, 'upserted_count', 0) or 0)
        total = len(operations)
        duplicates = total - inserted

        return {
            'enabled': True,
            'status': 'ok',
            'db': CONFIG['mongo_db'],
            'collection': CONFIG['mongo_collection'],
            'inserted': inserted,
            'duplicates': duplicates,
            'total': total,
        }
    except Exception as exc:
        hint = None
        if 'No such file or directory' in str(exc):
            hint = 'MONGO_URI com caminho local de certificado. No Colab remove tlsCAFile/ssl_ca_certs da URI.'
        return {
            'enabled': True,
            'status': 'error',
            'error': str(exc),
            'hint': hint,
            'db': CONFIG['mongo_db'],
            'collection': CONFIG['mongo_collection'],
        }
    finally:
        if client is not None:
            try:
                client.close()
            except Exception:
                pass


def save_to_tidb(df: pd.DataFrame, run_id: str) -> dict:
    if not CONFIG['tidb_enabled']:
        return {'enabled': False, 'status': 'disabled'}

    if pymysql is None:
        return {
            'enabled': True,
            'status': 'error',
            'error': 'pymysql is not installed. Install dependencies from requirements.txt.',
        }

    required = ['tidb_host', 'tidb_user', 'tidb_password', 'tidb_database', 'tidb_table']
    missing = [name for name in required if not CONFIG.get(name)]
    if missing:
        return {
            'enabled': True,
            'status': 'error',
            'error': f"Missing TiDB settings: {', '.join(missing)}",
        }

    records = json.loads(df.to_json(orient='records', force_ascii=False, date_format='iso'))
    if not records:
        return {'enabled': True, 'status': 'ok', 'written': 0, 'table_count': 0}

    ca_path = (CONFIG.get('tidb_ca_path') or '').strip()
    if ca_path and not os.path.exists(ca_path):
        if is_colab() and certifi is not None:
            ca_path = certifi.where()
        else:
            return {
                'enabled': True,
                'status': 'error',
                'error': f'TIDB_CA_PATH not found: {CONFIG.get("tidb_ca_path")}',
            }

    if not ca_path and 'tidbcloud' in str(CONFIG.get('tidb_host', '')).lower() and certifi is not None:
        ca_path = certifi.where()

    ssl_kwargs = {'ca': ca_path} if ca_path else None

    try:
        connection = pymysql.connect(
            host=CONFIG['tidb_host'],
            port=CONFIG['tidb_port'],
            user=CONFIG['tidb_user'],
            password=CONFIG['tidb_password'],
            autocommit=False,
            charset='utf8mb4',
            connect_timeout=CONFIG['tidb_connect_timeout'],
            read_timeout=30,
            write_timeout=30,
            ssl=ssl_kwargs,
        )
    except Exception as exc:
        return {
            'enabled': True,
            'status': 'error',
            'error': str(exc),
            'hint': 'No Colab usa TIDB_CA_PATH vazio ou /content/... e deixa o notebook usar certifi automaticamente.',
        }

    create_database_sql = f"CREATE DATABASE IF NOT EXISTS `{CONFIG['tidb_database']}`"
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS `{CONFIG['tidb_table']}` (
            `id` VARCHAR(64) NOT NULL,
            `run_id` VARCHAR(32) NOT NULL,
            `source_file` VARCHAR(255) NOT NULL,
            `ingested_at` DATETIME(6) NULL,
            `saved_at` DATETIME(6) NOT NULL,
            `payload_json` JSON NOT NULL,
            PRIMARY KEY (`id`),
            KEY `idx_run_id` (`run_id`),
            KEY `idx_source_file` (`source_file`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_bin
    """
    insert_sql = f"""
        INSERT INTO `{CONFIG['tidb_table']}`
            (`id`, `run_id`, `source_file`, `ingested_at`, `saved_at`, `payload_json`)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            `run_id` = VALUES(`run_id`),
            `source_file` = VALUES(`source_file`),
            `ingested_at` = VALUES(`ingested_at`),
            `saved_at` = VALUES(`saved_at`),
            `payload_json` = VALUES(`payload_json`)
    """

    rows = []
    saved_at = _utc_sql_datetime(None)
    for record in records:
        payload = dict(record)
        row_id = _stable_record_identity(payload)
        rows.append(
            (
                row_id,
                run_id,
                str(payload.get('_source_file', '')),
                _utc_sql_datetime(payload.get('_ingested_at')),
                saved_at,
                json.dumps(payload, ensure_ascii=False, default=str),
            )
        )

    try:
        with connection.cursor() as cursor:
            cursor.execute(create_database_sql)
            connection.select_db(CONFIG['tidb_database'])
            cursor.execute(create_table_sql)
            cursor.executemany(insert_sql, rows)
            connection.commit()
            cursor.execute(f"SELECT COUNT(*) FROM `{CONFIG['tidb_table']}`")
            table_count = int(cursor.fetchone()[0] or 0)

        return {
            'enabled': True,
            'status': 'ok',
            'db': CONFIG['tidb_database'],
            'table': CONFIG['tidb_table'],
            'written': len(rows),
            'table_count': table_count,
        }
    except Exception as exc:
        connection.rollback()
        return {
            'enabled': True,
            'status': 'error',
            'error': str(exc),
        }
    finally:
        connection.close()


def save_to_cratedb(df: pd.DataFrame, run_id: str) -> dict:
    if not CONFIG['cratedb_enabled']:
        return {'enabled': False, 'status': 'disabled'}

    if psycopg2 is None:
        return {
            'enabled': True,
            'status': 'error',
            'error': 'psycopg2-binary is not installed. Install dependencies from requirements.txt.',
        }

    required = ['cratedb_host', 'cratedb_user', 'cratedb_password', 'cratedb_database', 'cratedb_table']
    missing = [name for name in required if not CONFIG.get(name)]
    if missing:
        return {
            'enabled': True,
            'status': 'error',
            'error': f"Missing CrateDB settings: {', '.join(missing)}",
        }

    records = json.loads(df.to_json(orient='records', force_ascii=False, date_format='iso'))
    if not records:
        return {'enabled': True, 'status': 'ok', 'written': 0, 'table_count': 0}

    try:
        connection = psycopg2.connect(
            host=CONFIG['cratedb_host'],
            port=CONFIG['cratedb_port'],
            user=CONFIG['cratedb_user'],
            password=CONFIG['cratedb_password'],
            dbname=CONFIG['cratedb_database'],
            connect_timeout=CONFIG['tidb_connect_timeout'],
            sslmode=CONFIG['cratedb_sslmode'],
        )
    except Exception as exc:
        return {
            'enabled': True,
            'status': 'error',
            'error': str(exc),
            'db': CONFIG['cratedb_database'],
            'table': CONFIG['cratedb_table'],
        }

    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {CONFIG['cratedb_table']} (
            id VARCHAR PRIMARY KEY,
            run_id VARCHAR,
            source_file VARCHAR,
            ingested_at TIMESTAMP WITH TIME ZONE,
            saved_at TIMESTAMP WITH TIME ZONE,
            payload_json TEXT
        )
    """
    insert_sql = f"""
        INSERT INTO {CONFIG['cratedb_table']}
            (id, run_id, source_file, ingested_at, saved_at, payload_json)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (id) DO UPDATE SET
            run_id = excluded.run_id,
            source_file = excluded.source_file,
            ingested_at = excluded.ingested_at,
            saved_at = excluded.saved_at,
            payload_json = excluded.payload_json
    """

    rows = []
    saved_at = _utc_sql_datetime(None)
    for record in records:
        payload = dict(record)
        row_id = _stable_record_identity(payload)
        payload_json = json.dumps(payload, ensure_ascii=False, default=str)
        if len(payload_json) > 30000:
            payload_json = payload_json[:30000]
        rows.append(
            (
                row_id,
                run_id,
                str(payload.get('_source_file', '')),
                _utc_sql_datetime(payload.get('_ingested_at')),
                saved_at,
                payload_json,
            )
        )

    try:
        with connection.cursor() as cursor:
            cursor.execute(create_table_sql)
            cursor.executemany(insert_sql, rows)
            connection.commit()
            cursor.execute(f"SELECT COUNT(*) FROM {CONFIG['cratedb_table']}")
            table_count = int(cursor.fetchone()[0] or 0)

        return {
            'enabled': True,
            'status': 'ok',
            'db': CONFIG['cratedb_database'],
            'table': CONFIG['cratedb_table'],
            'written': len(rows),
            'table_count': table_count,
        }
    except Exception as exc:
        return {
            'enabled': True,
            'status': 'error',
            'error': str(exc),
            'db': CONFIG['cratedb_database'],
            'table': CONFIG['cratedb_table'],
        }
    finally:
        connection.close()

## 6) Email de resumo e anomalias


In [6]:
import html as ihtml

try:
    from google.colab import userdata  # type: ignore
except Exception:
    userdata = None

def _safe_float(value) -> float:
    try:
        return float(value)
    except Exception:
        return 0.0

def _build_profile_rows() -> list[tuple[str, float, float]]:
    rows: list[tuple[str, float, float]] = []
    for task, raw in clts.elapt.items():
        watch_secs = 0.0
        proc_secs = 0.0

        if isinstance(raw, dict):
            watch_secs = _safe_float(raw.get('tt', raw.get('watch', 0.0)))
            proc_secs = _safe_float(raw.get('tp', raw.get('proc', watch_secs)))
        elif isinstance(raw, (list, tuple)) and len(raw) >= 2:
            watch_secs = _safe_float(raw[0])
            proc_secs = _safe_float(raw[1])
        else:
            watch_secs = _safe_float(raw)
            proc_secs = watch_secs

        rows.append((str(task), watch_secs, proc_secs))

    return rows

def _build_html_table(rows: list[tuple[str, float, float]]) -> str:
    table_rows = []
    for task, watch_secs, proc_secs in rows:
        table_rows.append(
            "<tr>"
            f"<td style='border:1px solid #666;padding:4px 6px;text-align:left'>{ihtml.escape(task)}</td>"
            f"<td style='border:1px solid #666;padding:4px 6px;text-align:right'>{watch_secs:.2f}</td>"
            f"<td style='border:1px solid #666;padding:4px 6px;text-align:right'>{proc_secs:.2f}</td>"
            "</tr>"
        )

    head = (
        "<tr>"
        "<th style='border:1px solid #666;padding:4px 6px;text-align:left;background:#f2f2f2'>Task(s)</th>"
        "<th style='border:1px solid #666;padding:4px 6px;text-align:right;background:#f2f2f2'>watch time (secs)</th>"
        "<th style='border:1px solid #666;padding:4px 6px;text-align:right;background:#f2f2f2'>proc time (secs)</th>"
        "</tr>"
    )

    return (
        "<table style='border-collapse:collapse;font-family:Arial,Helvetica,sans-serif;font-size:13px;color:#222'>"
        + head
        + ''.join(table_rows)
        + "</table>"
    )


_build_profile_table_html = _build_html_table

def _build_anomaly_table_html(anomaly_report: dict | None) -> str:
    if not anomaly_report:
        return ""

    status = anomaly_report.get('status', 'skipped')
    anomalous_count = int(anomaly_report.get('anomalous_counters', 0))

    if status != 'ok' or anomalous_count == 0:
        return ""

    details = anomaly_report.get('details', [])
    if not details:
        return ""

    top_details = details[:10]

    table_rows = []
    for item in top_details:
        contador = ihtml.escape(str(item.get('contador', '')))
        current = int(item.get('current_count', 0))
        expected = float(item.get('expected_total_count', item.get('expected_count', 0.0)) or 0.0)
        ratio = item.get('ratio')
        ratio_str = f"{ratio*100:.1f}%" if ratio is not None else "N/A"
        missing = int(item.get('missing_total_count', item.get('missing_readings', 0)) or 0)
        lost_percentage = float(item.get('lost_total_percentage', item.get('lost_percentage', (missing / expected * 100) if expected else 0.0)) or 0.0)

        table_rows.append(
            "<tr>"
            f"<td style='border:1px solid #ccc;padding:4px 6px;text-align:left'>{contador}</td>"
            f"<td style='border:1px solid #ccc;padding:4px 6px;text-align:right'>{expected:.2f}</td>"
            f"<td style='border:1px solid #ccc;padding:4px 6px;text-align:right'>{current}</td>"
            f"<td style='border:1px solid #ccc;padding:4px 6px;text-align:right;color:red'>{ratio_str}</td>"
            f"<td style='border:1px solid #ccc;padding:4px 6px;text-align:right;color:#b00020'>{lost_percentage:.1f}%</td>"
            f"<td style='border:1px solid #ccc;padding:4px 6px;text-align:right;color:#d9534f'>-{missing}</td>"
            "</tr>"
        )

    head = (
        "<tr>"
        "<th style='border:1px solid #ccc;padding:4px 6px;text-align:left;background:#f9f9f9;color:#d9534f'>Contador</th>"
        "<th style='border:1px solid #ccc;padding:4px 6px;text-align:right;background:#f9f9f9;color:#d9534f'>Contagem esperada</th>"
        "<th style='border:1px solid #ccc;padding:4px 6px;text-align:right;background:#f9f9f9;color:#d9534f'>Atual</th>"
        "<th style='border:1px solid #ccc;padding:4px 6px;text-align:right;background:#f9f9f9;color:#d9534f'>Rácio %</th>"
        "<th style='border:1px solid #ccc;padding:4px 6px;text-align:right;background:#f9f9f9;color:#d9534f'>Percentagem perdida</th>"
        "<th style='border:1px solid #ccc;padding:4px 6px;text-align:right;background:#f9f9f9;color:#d9534f'>Soma em falta</th>"
        "</tr>"
    )

    table_html = (
        "<table style='border-collapse:collapse;font-family:Arial,Helvetica,sans-serif;font-size:12px;color:#222;margin-top:12px'>"
        + head
        + ''.join(table_rows)
        + "</table>"
    )

    intro = f"<p style='margin:12px 0 6px;color:#d9534f;font-weight:bold'>⚠ {anomalous_count} contador(es) com menos leituras do que o esperado (últimos 2 dias):</p>"
    return intro + table_html


def _send_email_via_brevo(subject: str, text_body: str, html_body: str, recipients: list[str]) -> dict:
    api_key = CONFIG.get('brevo_api_key', '').strip()
    sender_email = (CONFIG.get('brevo_sender_email') or CONFIG.get('email_from') or '').strip()
    sender_name = (CONFIG.get('brevo_sender_name') or 'PIP Water').strip()

    if not api_key:
        return {'status': 'error', 'error': 'BREVO_API_KEY is missing.'}
    if not sender_email:
        return {'status': 'error', 'error': 'BREVO_SENDER_EMAIL (or EMAIL_FROM) is missing.'}
    if not recipients:
        return {'status': 'error', 'error': 'No recipients configured.'}

    payload = {
        'sender': {'name': sender_name, 'email': sender_email},
        'to': [{'email': item} for item in recipients],
        'subject': subject,
        'textContent': text_body,
        'htmlContent': html_body,
    }

    response = requests.post(
        'https://api.brevo.com/v3/smtp/email',
        headers={
            'accept': 'application/json',
            'content-type': 'application/json',
            'api-key': api_key,
        },
        json=payload,
        timeout=CONFIG['request_timeout'],
    )

    if response.ok:
        return {'status': 'ok', 'provider': 'brevo', 'response': response.json() if response.content else {}}

    return {
        'status': 'error',
        'provider': 'brevo',
        'error': f'{response.status_code} {response.text[:500]}',
    }


def _send_email_via_smtp(subject: str, text_body: str, html_body: str, recipients: list[str]) -> dict:
    if not CONFIG['email_from'] or not CONFIG['email_username'] or not CONFIG['email_password']:
        return {'status': 'error', 'error': 'SMTP credentials are incomplete.'}

    message = MIMEMultipart('alternative')
    message['Subject'] = subject
    message['From'] = CONFIG['email_from']
    message['To'] = ', '.join(recipients)
    message.attach(MIMEText(text_body, 'plain'))
    message.attach(MIMEText(html_body, 'html'))

    ssl_context = ssl.create_default_context()
    smtp_host = CONFIG['email_smtp_host']
    smtp_port = CONFIG['email_smtp_port']

    if smtp_port == 465:
        with smtplib.SMTP_SSL(smtp_host, smtp_port, context=ssl_context) as server:
            server.login(CONFIG['email_username'], CONFIG['email_password'])
            server.sendmail(CONFIG['email_from'], recipients, message.as_string())
    else:
        with smtplib.SMTP(smtp_host, smtp_port) as server:
            server.starttls(context=ssl_context)
            server.login(CONFIG['email_username'], CONFIG['email_password'])
            server.sendmail(CONFIG['email_from'], recipients, message.as_string())

    return {'status': 'ok', 'provider': 'smtp', 'smtp_host': smtp_host, 'smtp_port': smtp_port}


def send_email_summary(result: dict | None) -> None: 
    if not CONFIG['email_enabled']:
        return

    recipients = [item.strip() for item in CONFIG['email_to'].split(',') if item.strip()]
    if not recipients:
        return

    status = result.get('status', 'unknown') if result else 'unknown'
    runtime_label = str(result.get('runtime') or detect_runtime() or 'unknown').strip().lower() if result else detect_runtime()
    subject = f"[PIP Water | {runtime_label}] {status.upper()} | {result.get('run_id', 'N/A') if result else 'N/A'}"
    anomalies = (result or {}).get('anomalies', {}) or {}
    anom_count = int(anomalies.get('anomalous_counters', 0) or 0)
    profile_rows = _build_profile_rows()
    profile_table_html = _build_profile_table_html(profile_rows)
    anomaly_table_html = _build_anomaly_table_html(anomalies)

    text_lines = [
        f"Environment: {runtime_label}",
        f"Run ID: {result.get('run_id', 'N/A') if result else 'N/A'}",
        f"Status: {status}",
        "",
    ]
    text_lines.extend(f"{task} | watch: {w:.2f} | proc: {p:.2f}" for task, w, p in profile_rows)

    if anom_count > 0:
        text_lines.append(f"\n[ANOMALIAS] {anom_count} contador(es) com problema nos últimos 2 dias")

    error_text = result.get('error') if result else None
    if error_text:
        text_lines.append(f"Error: {error_text}")

    text_body = "\n".join(text_lines) + "\nEsta é uma mensagem automática."

    html_body = (
        "<html><body style='font-family:Montserrat,Arial,sans-serif'>"
        f"<p><b>Environment:</b> {ihtml.escape(runtime_label)}</p>"
        f"<p><b>Run ID:</b> {ihtml.escape(str(result.get('run_id', 'N/A') if result else 'N/A'))}</p>"
        f"<p><b>Status:</b> {ihtml.escape(str(status).upper())}</p>"
        + profile_table_html
    )
    if anomaly_table_html:
        html_body += "<hr color=orange>" + anomaly_table_html
    if error_text:
        html_body += f"<hr color=orange><p style='color:#b00020'><b>Error:</b> {ihtml.escape(str(error_text))}</p>"
    html_body += "<hr color=orange>"
    html_body += "This message is an automated notification from PIP Water notebook pipeline"
    html_body += "</body></html>"

    preferred_backend = (CONFIG.get('email_backend') or 'smtp').strip().lower()
    if preferred_backend == 'brevo' or runtime_label == 'render' or is_render():
        send_result = _send_email_via_brevo(subject, text_body, html_body, recipients)
        if send_result.get('status') != 'ok' and preferred_backend == 'brevo' and is_render():
            raise RuntimeError(send_result.get('error', 'Brevo email send failed.'))
        if send_result.get('status') != 'ok':
            raise RuntimeError(send_result.get('error', 'Email send failed.'))
        return

    send_result = _send_email_via_smtp(subject, text_body, html_body, recipients)
    if send_result.get('status') != 'ok':
        raise RuntimeError(send_result.get('error', 'SMTP email send failed.'))

    # debug preview in notebook
    for task, w, p in _build_profile_rows():
        print(f"{task} | watch: {w:.2f} | proc: {p:.2f}")

# Note: call send_email_summary(result) from the pipeline to perform the send once

## 6) Tabela de comparação com os registos dos ultimos 2 dias


In [7]:
def _coerce_payload_dict(payload):
    if isinstance(payload, dict):
        return payload
    if isinstance(payload, (bytes, bytearray)):
        payload = payload.decode('utf-8', errors='ignore')
    if isinstance(payload, str):
        try:
            return json.loads(payload)
        except Exception:
            return {}
    return {}


def _flatten_meter_payload(record: dict, source_tag: str = '') -> list[dict]:
    header = record.get('header')
    body = record.get('body')
    if not isinstance(header, list) or not isinstance(body, list):
        return []

    columns = [str(col).strip() for col in header]
    rows = []

    for entry in body:
        if not isinstance(entry, (list, tuple)):
            continue

        mapped = {
            columns[idx]: entry[idx]
            for idx in range(min(len(columns), len(entry)))
        }
        device = str(
            mapped.get('Device')
            or mapped.get('device')
            or mapped.get('Contador')
            or mapped.get('contador')
            or ''
        ).strip()
        alias = str(mapped.get('Alias') or mapped.get('alias') or '').strip()
        timestamp_raw = (
            mapped.get('Date/Time')
            or mapped.get('timestamp')
            or mapped.get('Timestamp')
            or mapped.get('datahora')
        )
        timestamp = pd.to_datetime(timestamp_raw, dayfirst=True, errors='coerce')
        contador = device or alias

        if not contador:
            continue

        rows.append(
            {
                'contador': contador,
                'device': device,
                'alias': alias,
                'timestamp': timestamp,
                'source_file': record.get('_source_file', source_tag or ''),
                '_run_id': record.get('_run_id'),
            }
        )

    return rows


def _flatten_meter_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['contador', 'device', 'alias', 'timestamp', 'source_file', '_run_id'])

    flattened = []
    for record in df.to_dict(orient='records'):
        flattened.extend(_flatten_meter_payload(record))

    if not flattened:
        return pd.DataFrame(columns=['contador', 'device', 'alias', 'timestamp', 'source_file', '_run_id'])

    flat_df = pd.DataFrame(flattened)
    flat_df['timestamp'] = pd.to_datetime(flat_df['timestamp'], errors='coerce')
    return flat_df


def _load_tidb_history_for_anomalies(lookback_days: int) -> pd.DataFrame:
    if not CONFIG['tidb_enabled'] or pymysql is None:
        return pd.DataFrame(columns=['contador', 'device', 'alias', 'timestamp', 'source_file', '_run_id'])

    required = ['tidb_host', 'tidb_user', 'tidb_password', 'tidb_database', 'tidb_table']
    missing = [name for name in required if not CONFIG.get(name)]
    if missing:
        return pd.DataFrame(columns=['contador', 'device', 'alias', 'timestamp', 'source_file', '_run_id'])

    ca_path = (CONFIG.get('tidb_ca_path') or '').strip()
    if ca_path and not os.path.exists(ca_path):
        if is_colab() and certifi is not None:
            ca_path = certifi.where()
        else:
            return pd.DataFrame(columns=['contador', 'device', 'alias', 'timestamp', 'source_file', '_run_id'])

    if not ca_path and 'tidbcloud' in str(CONFIG.get('tidb_host', '')).lower() and certifi is not None:
        ca_path = certifi.where()

    ssl_kwargs = {'ca': ca_path} if ca_path else None
    cutoff = datetime.now(timezone.utc).replace(tzinfo=None) - pd.Timedelta(days=lookback_days)
    rows = []

    try:
        connection = pymysql.connect(
            host=CONFIG['tidb_host'],
            port=CONFIG['tidb_port'],
            user=CONFIG['tidb_user'],
            password=CONFIG['tidb_password'],
            autocommit=True,
            charset='utf8mb4',
            connect_timeout=CONFIG['tidb_connect_timeout'],
            read_timeout=30,
            write_timeout=30,
            ssl=ssl_kwargs,
        )
        connection.select_db(CONFIG['tidb_database'])
    except Exception:
        return pd.DataFrame(columns=['contador', 'device', 'alias', 'timestamp', 'source_file', '_run_id'])

    try:
        with connection.cursor() as cursor:
            cursor.execute(
                f"""
                SELECT `payload_json`, `saved_at`, `run_id`, `source_file`
                FROM `{CONFIG['tidb_table']}`
                WHERE `saved_at` >= %s
                ORDER BY `saved_at` ASC
                """,
                (cutoff,),
            )
            for payload_json, saved_at, run_id_value, source_file in cursor.fetchall():
                payload = _coerce_payload_dict(payload_json)
                if not payload:
                    continue
                payload['_db_saved_at'] = saved_at.isoformat() if saved_at else None
                payload['_run_id'] = run_id_value or payload.get('_run_id')
                payload['_source_file'] = source_file or payload.get('_source_file', '')
                rows.extend(_flatten_meter_payload(payload, source_tag=str(source_file or '')))
    except Exception:
        return pd.DataFrame(columns=['contador', 'device', 'alias', 'timestamp', 'source_file', '_run_id'])
    finally:
        connection.close()

    if not rows:
        return pd.DataFrame(columns=['contador', 'device', 'alias', 'timestamp', 'source_file', '_run_id'])

    history_df = pd.DataFrame(rows)
    history_df['timestamp'] = pd.to_datetime(history_df['timestamp'], errors='coerce')
    return history_df


def build_meter_anomaly_report(current_df: pd.DataFrame, run_id: str, lookback_days: int = 2) -> dict:
    threshold_ratio = float(CONFIG.get('anomaly_threshold_ratio', 0.8) or 0.8)
    current_flat = _flatten_meter_dataframe(current_df)

    if current_flat.empty:
        return {
            'status': 'skipped',
            'reason': 'No meter readings found in the current batch.',
            'lookback_days': lookback_days,
            'threshold_ratio': threshold_ratio,
            'run_id': run_id,
        }

    current_flat = current_flat.dropna(subset=['timestamp']).copy()
    if current_flat.empty:
        return {
            'status': 'skipped',
            'reason': 'Current batch has no parseable timestamps.',
            'lookback_days': lookback_days,
            'threshold_ratio': threshold_ratio,
            'run_id': run_id,
        }

    current_counts = (
        current_flat.groupby('contador', as_index=False)
        .agg(device=('device', 'first'), alias=('alias', 'first'), current_count=('contador', 'size'))
        .copy()
    )

    history_flat = _load_tidb_history_for_anomalies(lookback_days)
    if history_flat.empty:
        return {
            'status': 'skipped',
            'reason': f'No TiDB history found for the last {lookback_days} days.',
            'lookback_days': lookback_days,
            'threshold_ratio': threshold_ratio,
            'run_id': run_id,
            'current_readings': int(len(current_flat)),
            'current_counters': int(len(current_counts)),
        }

    history_flat = history_flat.dropna(subset=['timestamp']).copy()
    if history_flat.empty:
        return {
            'status': 'skipped',
            'reason': 'Historical TiDB rows have no parseable timestamps.',
            'lookback_days': lookback_days,
            'threshold_ratio': threshold_ratio,
            'run_id': run_id,
            'current_readings': int(len(current_flat)),
            'current_counters': int(len(current_counts)),
        }

    history_flat['day'] = history_flat['timestamp'].dt.floor('D')
    history_daily = history_flat.groupby(['contador', 'day'], as_index=False).size().rename(columns={'size': 'history_count'})
    history_stats = (
        history_daily.groupby('contador', as_index=False)
        .agg(
            history_avg=('history_count', 'mean'),
            history_median=('history_count', 'median'),
            history_min=('history_count', 'min'),
            history_max=('history_count', 'max'),
            history_days=('day', 'nunique'),
        )
        .copy()
    )

    comparison = history_stats.merge(current_counts, on='contador', how='left')
    comparison['current_count'] = comparison['current_count'].fillna(0).astype(int)
    comparison['expected_count'] = comparison['history_avg'].fillna(0.0)
    comparison['ratio'] = comparison.apply(
        lambda row: round(row['current_count'] / row['expected_count'], 3) if row['expected_count'] else None,
        axis=1,
    )
    comparison['is_anomaly'] = (
        comparison['expected_count'].gt(0)
        & comparison['current_count'].lt(comparison['expected_count'] * threshold_ratio)
    )

    flagged = comparison.loc[comparison['is_anomaly']].copy()
    flagged = flagged.sort_values(['ratio', 'current_count', 'contador'], ascending=[True, True, True])

    details = []
    for row in flagged.to_dict(orient='records'):
        expected_count = round(float(row.get('expected_count') or 0.0), 2)
        current_count = int(row.get('current_count') or 0)
        missing_count = max(0, int(round(expected_count - current_count)))
        details.append(
            {
                'contador': row.get('contador', ''),
                'device': row.get('device', ''),
                'alias': row.get('alias', ''),
                'current_count': current_count,
                'expected_count': expected_count,
                'missing_count': missing_count,
                'expected_total_count': expected_count,
                'missing_total_count': missing_count,
                'history_avg': round(float(row.get('history_avg') or 0.0), 2),
                'history_median': round(float(row.get('history_median') or 0.0), 2),
                'history_min': int(row.get('history_min') or 0),
                'history_max': int(row.get('history_max') or 0),
                'history_days': int(row.get('history_days') or 0),
                'ratio': row.get('ratio'),
                'missing_readings': missing_count,
            }
        )

    total_expected = float(comparison['expected_count'].sum() or 0.0)
    total_current = int(comparison['current_count'].sum() or 0)
    return {
        'status': 'ok',
        'source': 'tidb',
        'run_id': run_id,
        'lookback_days': lookback_days,
        'threshold_ratio': threshold_ratio,
        'current_readings': int(len(current_flat)),
        'current_counters': int(len(current_counts)),
        'history_readings': int(len(history_flat)),
        'history_counters': int(len(history_stats)),
        'expected_readings_total': round(total_expected, 2),
        'current_readings_total': total_current,
        'anomalous_counters': int(len(details)),
        'details': details,
    }

## 7) Execucao da pipeline


In [8]:
started = time.perf_counter()
phase_started = started
phase_seconds = {}
runtime = detect_runtime()
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
started_at_utc = datetime.now(timezone.utc).isoformat()


def close_phase(name: str) -> None:
    global phase_started
    now = time.perf_counter()
    phase_seconds[name] = round(now - phase_started, 4)
    phase_started = now



def publish_clts_profile(mongo_result: dict | None = None, tidb_result: dict | None = None, cratedb_result: dict | None = None) -> None:
    clts.elapt['Pipeline started'] = 0.0
    cumulative = 0.0
    steps = [
        ('file_discovery', 'File discovery'),
        ('download_and_normalize', 'Download and normalize'),
        ('transform', 'Transform'),
        ('save_outputs', 'Save outputs'),
        ('save_mongodb', 'Save MongoDB'),
        ('save_tidb', 'Save TiDB'),
        ('save_cratedb', 'Save CrateDB'),
    ]

    for step_name, label in steps:
        proc_time = float(phase_seconds.get(step_name, 0) or 0)
        cumulative += proc_time
        clts.elapt[label] = round(cumulative, 2)

        if step_name == 'save_mongodb' and mongo_result:
            if mongo_result.get('enabled', False):
                if mongo_result.get('status') == 'ok':
                    clts.elapt[
                        f"... [mongodb] {mongo_result.get('collection', CONFIG['mongo_collection'])}: "
                        f"inserted {mongo_result.get('inserted', 0)}, skipped {mongo_result.get('duplicates', 0)}"
                    ] = round(cumulative, 2)
                elif mongo_result.get('status') == 'skipped':
                    clts.elapt['... [mongodb] skipped'] = round(cumulative, 2)
                else:
                    clts.elapt[f"... [mongodb] error: {mongo_result.get('error', 'unknown')}"] = round(cumulative, 2)

        if step_name == 'save_tidb' and tidb_result:
            if tidb_result.get('enabled', False):
                if tidb_result.get('status') == 'ok':
                    clts.elapt[
                        f"... [tidb] {tidb_result.get('table', CONFIG['tidb_table'])}: "
                        f"written {tidb_result.get('written', 0)}, rows in table {tidb_result.get('table_count', 0)}"
                    ] = round(cumulative, 2)
                elif tidb_result.get('status') == 'skipped':
                    clts.elapt['... [tidb] skipped'] = round(cumulative, 2)
                else:
                    clts.elapt[f"... [tidb] error: {tidb_result.get('error', 'unknown')}"] = round(cumulative, 2)

        if step_name == 'save_cratedb' and cratedb_result:
            if cratedb_result.get('enabled', False):
                if cratedb_result.get('status') == 'ok':
                    clts.elapt[
                        f"... [cratedb] {cratedb_result.get('table', CONFIG['cratedb_table'])}: "
                        f"written {cratedb_result.get('written', 0)}, rows in table {cratedb_result.get('table_count', 0)}"
                    ] = round(cumulative, 2)
                elif cratedb_result.get('status') == 'skipped':
                    clts.elapt['... [cratedb] skipped'] = round(cumulative, 2)
                else:
                    clts.elapt[f"... [cratedb] error: {cratedb_result.get('error', 'unknown')}"] = round(cumulative, 2)

    clts.elapt['Overall pipeline'] = round(cumulative, 2)


print('=' * 60)
print('PIP WATER | NOTEBOOK')
print('=' * 60)

try:
    clts.elapt['Starting file discovery'] = clts.deltat(_tstart)
    files = list_from_json_file_urls()
    mode = 'JSON_FILE_URLS'

    if not files:
        files = list_from_local_dir()
        mode = 'LOCAL_JSON_DIR'

    if not files:
        files = list_from_known_names()
        mode = 'REPO_JSON_FILES'

    if not files:
        files = list_from_github(TOKEN)
        mode = 'GITHUB_API'

    files = apply_date_window(files)

    if mode in {'REPO_JSON_FILES', 'GITHUB_API'}:
        sample_size = int(CONFIG.get('repo_sample_size', 30) or 30)
        if sample_size > 0 and len(files) > sample_size:
            files = random.sample(files, sample_size)

    max_files = int(CONFIG.get('max_files', 0) or 0)
    if max_files > 0:
        files = files[:max_files]

    close_phase('file_discovery')

    if not files:
        raise RuntimeError('No JSON files found to process.')

    if CONFIG['verbose']:
        print('Runtime:', runtime)
        print('Read mode:', mode)
        print('Files selected:', len(files))

    all_records = []
    files_ok = 0
    files_error = 0

    for item in files:
        try:
            data = read_json_source(item['source'], TOKEN)
            all_records.extend(normalize_records(data, item['name'], run_id))
            files_ok += 1
            if CONFIG['verbose']:
                print(f"  OK  {item['name']}")
        except Exception as exc:
            files_error += 1
            print(f"  ERR {item['name']}: {exc}")

    close_phase('download_and_normalize')

    df = pd.DataFrame(all_records)
    df = light_clean(df)
    close_phase('transform')

    anomaly_report = build_meter_anomaly_report(df, run_id, lookback_days=2)
    anomaly_path = Path(CONFIG['output_folder']) / 'water_anomaly_report.json'
    Path(CONFIG['output_folder']).mkdir(parents=True, exist_ok=True)
    anomaly_path.write_text(json.dumps(anomaly_report, indent=2, ensure_ascii=False), encoding='utf-8')

    if CONFIG['verbose']:
        print('Anomaly report status:', anomaly_report.get('status'))
        print('Anomalous counters:', anomaly_report.get('anomalous_counters', 0))

    outputs = save_outputs(df)
    outputs['anomaly_report_path'] = str(anomaly_path)
    close_phase('save_outputs')

    if CONFIG.get('skip_db_writes', False):
        mongo_result = {'enabled': False, 'status': 'skipped', 'reason': 'SKIP_DB_WRITES=true'}
        tidb_result = {'enabled': False, 'status': 'skipped', 'reason': 'SKIP_DB_WRITES=true'}
        cratedb_result = {'enabled': False, 'status': 'skipped', 'reason': 'SKIP_DB_WRITES=true'}
        close_phase('save_mongodb')
        close_phase('save_tidb')
        close_phase('save_cratedb')
    else:
        mongo_result = save_to_mongodb(df, run_id)
        close_phase('save_mongodb')

        tidb_result = save_to_tidb(df, run_id)
        close_phase('save_tidb')

        cratedb_result = save_to_cratedb(df, run_id)
        close_phase('save_cratedb')

    publish_clts_profile(mongo_result, tidb_result, cratedb_result)

    elapsed = round(time.perf_counter() - started, 2)
    result = {
        'status': 'ok',
        'runtime': runtime,
        'run_id': run_id,
        'started_at_utc': started_at_utc,
        'mode': mode,
        'files_selected': len(files),
        'files_ok': files_ok,
        'files_error': files_error,
        'records': len(df),
        'elapsed_seconds': elapsed,
        'performance': {'phase_seconds': phase_seconds},
        'output': outputs,
        'mongodb': mongo_result,
        'tidb': tidb_result,
        'cratedb': cratedb_result,
        'anomalies': anomaly_report,
        'sample': df.head(5).to_dict(orient='records'),
    }

    Path(outputs['result_path']).write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')

except Exception as exc:
    elapsed = round(time.perf_counter() - started, 2)
    publish_clts_profile()
    result = {
        'status': 'error',
        'runtime': runtime,
        'run_id': run_id,
        'started_at_utc': started_at_utc,
        'error': str(exc),
        'hint': (
            'Use GITHUB_TOKEN com permissao de leitura no repo privado, ou define '
            'LOCAL_JSON_DIR, JSON_FILE_URLS, ou REPO_JSON_FILES.'
        ),
        'elapsed_seconds': elapsed,
        'performance': {'phase_seconds': phase_seconds},
    }
    outputs = {'result_path': str(Path(CONFIG['output_folder']) / 'water_result.json')}
    Path(outputs['result_path']).write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')

print(json.dumps(result, indent=2, ensure_ascii=False))

try:
    send_email_summary(result)
    if CONFIG['email_enabled']:
        print('Resumo por email: tentativa efetuada.')
except Exception as email_exc:
    print('Falha no envio de email:', email_exc)


PIP WATER | NOTEBOOK
Runtime: local
Read mode: GITHUB_API
Files selected: 30
  OK  sensors_202511211730281(1).json
  OK  sensors_202511211654446.json
  OK  sensors_202511171210529.json
  OK  sensors_202510271520069.json
  OK  sensors_202510281255080.json
  OK  sensors_202511191732575.json
  OK  sensors_202511171252250.json
  OK  sensors_202510281303290.json
  OK  sensors_202510291641539.json
  OK  sensors_202510281242269.json
  OK  sensors_202510281202302.json
  OK  sensors_202510301139081.json
  OK  sensors_202510281316062 (1).json
  OK  sensors_202510271351257.json
  OK  sensors_202511171257174.json
  OK  sensors_202511191941346.json
  OK  sensors_202510281245577.json
  OK  sensors_202510291615042.json
  OK  sensors_202511211453499.json
  OK  sensors_202511171219564.json
  OK  sensors_202510271617168.json
  OK  sensors_202510241459581.json
  OK  sensors_202511211725141(2).json
  OK  sensors_202510271329517.json
  OK  sensors_202510281320214.json
  OK  sensors_202510301457013.json
  O

C:\Users\henrique\AppData\Local\Temp\ipykernel_23024\4245102509.py:84: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow()


{
  "status": "ok",
  "runtime": "local",
  "run_id": "20260601T214013Z",
  "started_at_utc": "2026-06-01T21:40:13.793114+00:00",
  "mode": "GITHUB_API",
  "files_selected": 30,
  "files_ok": 30,
  "files_error": 0,
  "records": 30,
  "elapsed_seconds": 21.45,
  "performance": {
    "phase_seconds": {
      "file_discovery": 1.0275,
      "download_and_normalize": 13.7316,
      "transform": 0.0118,
      "save_outputs": 0.0614,
      "save_mongodb": 1.7721,
      "save_tidb": 4.0035,
      "save_cratedb": 0.846
    }
  },
  "output": {
    "json_path": "output_water\\water_data_raw.json",
    "csv_path": "output_water\\water_data_clean.csv",
    "result_path": "output_water\\water_result.json",
    "anomaly_report_path": "output_water\\water_anomaly_report.json"
  },
  "mongodb": {
    "enabled": true,
    "status": "ok",
    "db": "pi_water",
    "collection": "water_records",
    "inserted": 0,
    "duplicates": 30,
    "total": 30
  },
  "tidb": {
    "enabled": true,
    "status":

## 8) Validação rapida


In [9]:
result_path = Path(CONFIG['output_folder']) / 'water_result.json'

if not result_path.exists():
    print('Nao foi gerado water_result.json')
else:
    print('Result file:', result_path)
    try:
        result_obj = json.loads(result_path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('Erro a ler/parsing do water_result.json:', exc)
        result_obj = None

    if not result_obj:
        print('Resultado indisponivel (pipeline pode ainda estar a escrever o ficheiro).')
    elif result_obj.get('status') != 'ok':
        print('Erro:', result_obj.get('error'))
        print('Hint:', result_obj.get('hint'))
        if result_obj.get('mongodb'):
            print('MongoDB:', result_obj.get('mongodb'))
        if result_obj.get('tidb'):
            print('TiDB:', result_obj.get('tidb'))
    else:
        csv_path = Path(result_obj['output']['csv_path'])
        if not csv_path.exists():
            print('CSV ainda nao existe:', csv_path)
        else:
            try:
                total_rows = len(pd.read_csv(csv_path))
                preview = pd.read_csv(csv_path, nrows=5)
                print('Rows:', total_rows)
                print('Preview (5 rows):')
                print(preview.to_string(index=False))
            except Exception as exc:
                print('Erro a ler CSV:', exc)

SyntaxError: '(' was never closed (2966815007.py, line 19)

## 9) Opcional: modo local por upload
Se o acesso ao repo privado falhar no Colab, faz upload de JSON para /content/data/aqualog e define LOCAL_JSON_DIR.

In [ ]:
# Opcional: fallback com upload manual de JSON
# from google.colab import files
# os.makedirs('/content/data/aqualog', exist_ok=True)
# uploaded = files.upload()
# for fname, content in uploaded.items():
#     if fname.lower().endswith('.json'):
#         with open(f'/content/data/aqualog/{fname}', 'wb') as fh:
#             fh.write(content)
# os.environ['LOCAL_JSON_DIR'] = '/content/data/aqualog'
# CONFIG['local_json_dir'] = '/content/data/aqualog'
# print('LOCAL_JSON_DIR configurado.')

## 🧪 TEST: Generate Anomaly Test Data

To test the anomaly detection feature, run the cell below to generate a JSON file with one meter showing anomalous readings (much fewer readings than expected).

In [ ]:
# Generate test JSON with anomalous meter directly
test_json_path = Path('output_water/test_anomaly_sensors.json')

# Standard header
header = ["Device", "Alias", "Date/Time", "Value (l)", "Reading (l)"]

# Generate 10 normal meters with ~20 readings each
normal_meters = {
    "D13XF126371P": ("2081", 19642880),      # This will be ANOMALOUS
    "D14XF111533Z": ("148391", 2500000),
    "D15XF224455A": ("2089", 1800000),
    "D16XF333666B": ("2090", 3200000),
    "D17XF444777C": ("2091", 2100000),
    "D18XF555888D": ("2092", 1500000),
    "D19XF666999E": ("2093", 2800000),
    "D20XF777000F": ("2094", 1900000),
    "D21XF888111G": ("2095", 2400000),
    "D22XF999222H": ("2096", 1700000),
}

now = datetime.now(timezone.utc).replace(tzinfo=None)
body = []

# Generate readings for normal meters
for meter_id, (alias, base_reading) in normal_meters.items():
    num_readings = 3 if meter_id == "D13XF126371P" else 20  # ANOMALOUS: only 3 readings
    
    for i in range(num_readings):
        timestamp = now - timedelta(hours=30-i)
        value_l = 50 * (i + 1)
        reading = base_reading + value_l
        body.append([
            meter_id,
            alias,
            timestamp.strftime("%d/%m/%Y %H:%M"),
            f"{value_l:,}",
            f"{reading:,}"
        ])

data = {
    "header": header,
    "body": body,
    "footer": None
}

# Write to file
test_json_path.parent.mkdir(parents=True, exist_ok=True)
with open(test_json_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✅ Test file created successfully!")
print(f"   Location: {test_json_path}")
print(f"   Total readings: {len(body)}")

# Count readings per meter
meter_counts = {}
for row in body:
    meter = row[0]
    meter_counts[meter] = meter_counts.get(meter, 0) + 1

print(f"\n📊 Readings per meter:")
for meter, count in sorted(meter_counts.items()):
    status = "⚠️ ANOMALY (only 3)" if count == 3 else f"✓ Normal ({count})"
    print(f"   {meter}: {status}")

print(f"\n🔧 NEXT STEPS to test anomalies:")
print(f"   1. Add to REPO_JSON_FILES in Discovery cell (Cell 1):")
print(f"      'output_water/test_anomaly_sensors.json'")
print(f"   2. Re-run the main pipeline (Cell 17) to see anomalies in email ⚠️")

✅ Test file created successfully!
   Location: output_water\test_anomaly_sensors.json
   Total readings: 183

📊 Readings per meter:
   D13XF126371P: ⚠️ ANOMALY (only 3)
   D14XF111533Z: ✓ Normal (20)
   D15XF224455A: ✓ Normal (20)
   D16XF333666B: ✓ Normal (20)
   D17XF444777C: ✓ Normal (20)
   D18XF555888D: ✓ Normal (20)
   D19XF666999E: ✓ Normal (20)
   D20XF777000F: ✓ Normal (20)
   D21XF888111G: ✓ Normal (20)
   D22XF999222H: ✓ Normal (20)

🔧 NEXT STEPS to test anomalies:
   1. Add to REPO_JSON_FILES in Discovery cell (Cell 1):
      'output_water/test_anomaly_sensors.json'
   2. Re-run the main pipeline (Cell 17) to see anomalies in email ⚠️


## 🔧 Run Pipeline with Anomaly Test Data

Now add the test file to the discovery and run the pipeline to see anomalies in the email:

In [ ]:
# Setup clean test directory with only test sensor data
import shutil

print("=" * 60)
print("TESTING ANOMALY DETECTION - CLEAN SETUP")
print("=" * 60)

# Create a clean test directory
test_dir = Path('test_data_anomaly')
test_dir.mkdir(exist_ok=True)

# Copy only the test_anomaly_sensors.json file (clean, no old report files)
test_file_src = Path('output_water/test_anomaly_sensors.json')
test_file_dst = test_dir / 'test_anomaly_sensors.json'

if test_file_src.exists():
    shutil.copy(test_file_src, test_file_dst)
    print(f"✅ Copied test file to clean directory: {test_file_dst}")
else:
    print(f"❌ Test file not found: {test_file_src}")

# Configure to use the clean test directory
CONFIG['local_json_dir'] = str(test_dir)
CONFIG['json_file_urls'] = ''
CONFIG['repo_json_files'] = ''

print(f"\n📝 Discovery configuration:")
print(f"   LOCAL_JSON_DIR: {test_dir}")

print(f"\n📊 Test file contents:")
with open(test_file_dst) as f:
    test_data = json.load(f)

meter_counts = {}
for row in test_data.get('body', []):
    meter = row[0]
    meter_counts[meter] = meter_counts.get(meter, 0) + 1

for meter, count in sorted(meter_counts.items()):
    status = "⚠️ ANOMALY (3 readings only)" if count == 3 else f"✓ Normal ({count} readings)"
    print(f"   {meter}: {status}")

print(f"\n🚀 Now run Cell 17 to see anomalies appear in the email!")


TESTING ANOMALY DETECTION - CLEAN SETUP
✅ Copied test file to clean directory: test_data_anomaly\test_anomaly_sensors.json

📝 Discovery configuration:
   LOCAL_JSON_DIR: test_data_anomaly

📊 Test file contents:
   D13XF126371P: ⚠️ ANOMALY (3 readings only)
   D14XF111533Z: ✓ Normal (20 readings)
   D15XF224455A: ✓ Normal (20 readings)
   D16XF333666B: ✓ Normal (20 readings)
   D17XF444777C: ✓ Normal (20 readings)
   D18XF555888D: ✓ Normal (20 readings)
   D19XF666999E: ✓ Normal (20 readings)
   D20XF777000F: ✓ Normal (20 readings)
   D21XF888111G: ✓ Normal (20 readings)
   D22XF999222H: ✓ Normal (20 readings)

🚀 Now run Cell 17 to see anomalies appear in the email!


## 📧 Email Preview - Anomaly Table



In [ ]:
# Display preview of what the email would show
anomaly_data = result.get('anomalies', {})

print("\n" + "=" * 60)
print("EMAIL PREVIEW: ANOMALY TABLE")
print("=" * 60)

anomalous_count = int(anomaly_data.get('anomalous_counters', 0))
if anomalous_count == 0:
    print("\n✅ No anomalies detected - email would NOT include anomaly table")
else:
    print(f"\n⚠️  {anomalous_count} contador(es) com anomalias detectadas!")
    print(f"   Status: {anomaly_data.get('status')}")
    print(f"   Threshold: {anomaly_data.get('threshold_ratio', 0.8)*100:.0f}%")
    
    details = anomaly_data.get('details', [])
    print(f"\n📊 TOP 10 ANOMALIAS (showing in email):\n")
    
    # Show in table format
    print(f"{'Contador':<20} {'Esperado':<10} {'Atual':<8} {'Rácio %':<10} {'Faltas':<8}")
    print("-" * 60)
    
    for item in details[:10]:
        contador = str(item.get('contador', ''))[:19]
        expected = int(item.get('expected_count', 0))
        current = int(item.get('current_count', 0))
        ratio = item.get('ratio', 0)
        ratio_pct = f"{ratio*100:.1f}%"
        missing = int(item.get('missing_readings', 0))
        
        print(f"{contador:<20} {expected:<10} {current:<8} {ratio_pct:<10} -{missing:<7}")
    
    if len(details) > 10:
        print(f"\n   ... e mais {len(details)-10} contadores com problema")

print("\n✅ O email incluiria uma tabela (HTML) com estas anomalias!")
print("   Ordenadas por pior situação (ratio crescente)")
print("   Top 10 mostradas, resto disponível em water_anomaly_report.json")



EMAIL PREVIEW: ANOMALY TABLE

⚠️  8 contador(es) com anomalias detectadas!
   Status: ok
   Threshold: 80%

📊 TOP 10 ANOMALIAS (showing in email):

Contador             Esperado   Atual    Rácio %    Faltas  
------------------------------------------------------------
D15XF224455A         10         0        0.0%       -10     
D16XF333666B         10         0        0.0%       -10     
D17XF444777C         10         0        0.0%       -10     
D18XF555888D         10         0        0.0%       -10     
D19XF666999E         10         0        0.0%       -10     
D20XF777000F         10         0        0.0%       -10     
D21XF888111G         10         0        0.0%       -10     
D22XF999222H         10         0        0.0%       -10     

✅ O email incluiria uma tabela (HTML) com estas anomalias!
   Ordenadas por pior situação (ratio crescente)
   Top 10 mostradas, resto disponível em water_anomaly_report.json
